<a href="https://colab.research.google.com/github/vincentmartin/tp-fine-tuning-student-version/blob/main/tp-fine-tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP fine-tuning de LLM avec LoRA

Dans le TP d'initiation, vous avez piloté un LLM uniquement par le **prompt**. Dans ce TP, vous allez modifier son **comportement** en l'entraînant sur vos propres données : c'est le *fine-tuning* supervisé (SFT, *Supervised Fine-Tuning*), réalisé avec la méthode **LoRA** sur le GPU gratuit de Colab.

En sortie de ce TP, vous serez capable de :
- Choisir entre prompt engineering, RAG et fine-tuning, et estimer la mémoire nécessaire à un entraînement
- Préparer un jeu de données d'instructions au format *prompt / completion* avec le *chat template* du modèle
- Expliquer le masquage de la perte (*loss masking*) : sur quels tokens le modèle apprend-il ?
- Configurer LoRA (rang, alpha, modules ciblés) et calculer le nombre de paramètres entraînés
- Entraîner avec la bibliothèque TRL (`SFTTrainer`) et lire une courbe de perte
- Évaluer rigoureusement le modèle fine-tuné face au modèle de base (ROUGE, exemples, oubli catastrophique)
- Sauvegarder, recharger et fusionner un adaptateur LoRA
- (Mini-projet) Appliquer QLoRA à un modèle plus gros, ou étudier l'effet de la quantité de données

**Modèle utilisé** : [Qwen3-1.7B](https://huggingface.co/Qwen/Qwen3-1.7B) (Apache 2.0), le même que dans le TP d'initiation. **Tâche** : résumer des dialogues ([DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum)) dans le style des résumés humains du jeu de données.

### Instructions pour l'exécution sur Google Colab

Aller dans `Exécution -> Modifier le type d'exécution` puis sélectionner `T4 GPU`.

![Colab GPU](https://raw.githubusercontent.com/vincentmartin/tp-fine-tuning-student-version/main/resources/colab_gpu.png "T4-GPU")

**Pas de GPU disponible sur Colab ?** (quota atteint, heure de pointe) : utilisez [Kaggle Notebooks](https://www.kaggle.com/code), gratuit avec environ 30 h de GPU par semaine (`Settings -> Accelerator -> GPU T4 x2`, et activer `Internet` après vérification du compte par téléphone). Importez ce notebook via `File -> Import Notebook`.

**Durée des calculs** : l'entraînement principal prend une dizaine de minutes sur un T4. Pendant qu'il tourne, avancez sur les questions. Colab coupe une session inactive : gardez l'onglet ouvert.

**Assistants IA** : vous pouvez utiliser l'assistant intégré à Colab, mais vous devez être capables d'expliquer **chaque ligne** du code que vous rendez. Les questions d'analyse comptent autant que le code.

## 0. Installation et configuration

Colab fournit déjà `torch` : **ne le réinstallez pas**, cela casserait la compatibilité avec CUDA. Nous installons ou mettons à jour :
- **transformers**, **accelerate**, **datasets** : modèles, placement sur GPU, jeux de données
- **peft** : *Parameter-Efficient Fine-Tuning*, dont LoRA
- **trl** : *Transformer Reinforcement Learning*, la bibliothèque d'entraînement de Hugging Face (SFT, DPO, GRPO...)
- **bitsandbytes** : quantification 4 bits (QLoRA, mini-projet A)
- **evaluate** / **rouge_score** : métriques d'évaluation

In [ ]:
%pip install -q -U transformers accelerate datasets peft trl bitsandbytes evaluate rouge_score

**Remarque** : si l'import ci-dessous échoue juste après l'installation, redémarrez la session (`Exécution -> Redémarrer la session`) puis relancez les cellules à partir d'ici.

In [ ]:
import gc
import os
import time

import evaluate
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import peft
import torch
import transformers
import trl
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed

print(f"torch {torch.__version__} | transformers {transformers.__version__} | peft {peft.__version__} | trl {trl.__version__}")

assert torch.cuda.is_available(), "Pas de GPU : vérifiez le type d'exécution (T4 GPU). Le fine-tuning sur CPU est inenvisageable."
print("GPU :", torch.cuda.get_device_name(0))
# Le T4 (architecture Turing, compute capability 7.5) ne gère pas nativement le bfloat16 :
# on utilise float16 sur ce type de GPU, bfloat16 sur les GPU récents (Ampere et +).
USE_BF16 = torch.cuda.get_device_capability()[0] >= 8
DTYPE = torch.bfloat16 if USE_BF16 else torch.float16


def gpu_memory():
    """Mémoire GPU actuellement allouée et pic depuis le dernier reset, en Go."""
    return f"{torch.cuda.memory_allocated() / 1e9:.2f} Go alloués (pic : {torch.cuda.max_memory_allocated() / 1e9:.2f} Go)"


def free_memory():
    """Libère la mémoire GPU des objets Python supprimés (à appeler après `del model`)."""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()


set_seed(42)

## 1. Pourquoi (et quand) fine-tuner ?

Pour adapter un LLM à un besoin, il existe trois leviers, du moins coûteux au plus coûteux :

| Levier | Ce qu'il change | Quand l'utiliser |
|---|---|---|
| **Prompt engineering** (TP d'initiation) | Les instructions et exemples donnés au modèle | Toujours en premier : rapide, gratuit, réversible |
| **RAG** | Les **connaissances** fournies dans le contexte | Le modèle doit répondre à partir de documents (internes, récents) |
| **Fine-tuning** | Les **poids** du modèle, donc son **comportement** | Un style, un format ou une tâche précise à reproduire de façon fiable, un prompt trop long à répéter, un petit modèle à rendre aussi bon qu'un gros sur **une** tâche |

Le fine-tuning est un mauvais outil pour ajouter des connaissances (le modèle hallucine sur ce qu'il a vu une seule fois) : pour cela, on préfère le RAG.

### Fine-tuning complet ou PEFT ?

Entraîner **tous** les poids d'un modèle (*full fine-tuning*) demande beaucoup de mémoire. Avec l'optimiseur Adam en précision mixte, il faut stocker pour **chaque paramètre** :
- les poids en 16 bits (2 octets) et leur gradient (2 octets) ;
- une copie des poids en 32 bits (4 octets) ;
- les deux moments d'Adam en 32 bits (2 × 4 octets).

Soit environ **16 octets par paramètre**, sans compter les activations.

**Exercice 1** : écrire la fonction `training_memory_gb(n_params, bytes_per_param)` et l'appliquer à Qwen3-1.7B (1,7 milliard de paramètres) et à un modèle de 8 milliards. Comparer à la mémoire d'un GPU T4 (15 Go).

In [ ]:
def training_memory_gb(n_params, bytes_per_param=16):
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


for name, n_params in {"Qwen3-1.7B": 1.7e9, "Llama-3.1-8B": 8e9}.items():
    print(f"{name:12} : inférence fp16 ≈ {training_memory_gb(n_params, 2):5.1f} Go | fine-tuning complet ≈ {training_memory_gb(n_params):6.1f} Go")

Le fine-tuning complet est donc hors de portée d'un T4, même pour un petit modèle. La solution est le **PEFT** (*Parameter-Efficient Fine-Tuning*) : on **gèle** les poids d'origine et on n'entraîne qu'un petit nombre de paramètres ajoutés. La méthode PEFT la plus utilisée est **LoRA** (*Low-Rank Adaptation*), que nous détaillons en partie 4.

**Question** : avec LoRA, seuls environ 1 % des paramètres sont entraînés. Que deviennent les 16 octets par paramètre pour les 99 % restants ? Quelle est alors, approximativement, la mémoire nécessaire pour Qwen3-1.7B ?

✍️ **Votre réponse :**



## 2. La tâche et le modèle de base

### Le jeu de données

[DialogSum](https://huggingface.co/datasets/knkarthick/dialogsum) contient environ 13 000 dialogues de la vie courante en anglais, chacun accompagné d'un résumé écrit par un humain. Les résumés suivent des **conventions précises** : les interlocuteurs sont désignés par `#Person1#` et `#Person2#`, le résumé tient en une ou deux phrases, au présent ou au passé.

In [ ]:
dataset = load_dataset("knkarthick/dialogsum")
print(dataset)
example = dataset["train"][0]
print("\n--- DIALOGUE ---\n" + example["dialogue"])
print("\n--- RÉSUMÉ HUMAIN ---\n" + example["summary"])

Pour l'évaluation, nous utilisons le split `test`. Attention : chaque dialogue de ce split y apparaît **3 fois**, avec 3 résumés humains différents. Nous regroupons donc les références par dialogue et évaluons sur des dialogues **distincts**, en comparant chaque prédiction aux 3 références (ROUGE prend alors le meilleur score).

In [ ]:
N_EVAL = 50  # nombre de dialogues de test distincts utilisés pour l'évaluation

test_df = dataset["test"].to_pandas()
eval_df = test_df.groupby("dialogue", sort=False)["summary"].apply(list).reset_index().head(N_EVAL)
eval_dialogues = eval_df["dialogue"].tolist()
eval_references = eval_df["summary"].tolist()
print(f"{len(test_df)} lignes de test -> {test_df['dialogue'].nunique()} dialogues distincts, {len(eval_dialogues)} utilisés")

### Le modèle de base

Nous chargeons Qwen3-1.7B en 16 bits. C'est un modèle *instruct* : il sait déjà résumer, sans aucun entraînement de notre part. La question est donc : **le fine-tuning apporte-t-il quelque chose par rapport à un bon modèle généraliste ?**

In [ ]:
MODEL_NAME = "Qwen/Qwen3-1.7B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, dtype=DTYPE, device_map="cuda")
print(f"{model.num_parameters() / 1e9:.2f} milliards de paramètres | {gpu_memory()}")

Les deux fonctions ci-dessous sont fournies. `build_messages` construit la conversation envoyée au modèle ; `summarize` génère les résumés **par lots** (*batch*), ce qui est bien plus rapide qu'un dialogue à la fois.

Notez deux détails importants pour un modèle *decoder-only* :
- le *padding* se fait **à gauche** pour la génération par lots, afin que tous les prompts se terminent au même endroit et que la génération démarre juste après ;
- on ne décode que les **nouveaux tokens**, sinon la sortie contient aussi le prompt (une erreur fréquente qui fausse complètement l'évaluation).

In [ ]:
INSTRUCTION = "Summarize the following conversation."


def build_messages(dialogue):
    return [{"role": "user", "content": f"{INSTRUCTION}\n\n{dialogue}"}]


@torch.no_grad()
def summarize(model, dialogues, batch_size=16, max_new_tokens=128):
    """Résume une liste de dialogues en décodage glouton (greedy), sans mode raisonnement."""
    model.eval()
    tokenizer.padding_side = "left"
    summaries = []
    for start in range(0, len(dialogues), batch_size):
        prompts = [
            tokenizer.apply_chat_template(build_messages(d), tokenize=False, add_generation_prompt=True, enable_thinking=False)
            for d in dialogues[start:start + batch_size]
        ]
        inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, temperature=None, top_p=None, top_k=None)
        new_tokens = outputs[:, inputs["input_ids"].shape[1]:]
        summaries += [s.strip() for s in tokenizer.batch_decode(new_tokens, skip_special_tokens=True)]
    return summaries


print(summarize(model, [example["dialogue"]])[0])

**Exercice 2** : générer les résumés du modèle de base sur les `N_EVAL` dialogues d'évaluation (variable `base_summaries`), puis écrire la fonction `rouge_scores(predictions, references)` qui renvoie les scores ROUGE en pourcentage. Le module `evaluate` accepte plusieurs références par prédiction (une liste de listes).

In [ ]:
rouge = evaluate.load("rouge")


def rouge_scores(predictions, references):
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


start = time.time()
# VOTRE CODE ICI
print(f"Génération : {time.time() - start:.0f} s")
base_scores = rouge_scores(base_summaries, eval_references)
base_scores

**Exercice 3** : comparer la **longueur moyenne** (en mots) des résumés du modèle de base et des résumés humains, et afficher trois exemples côte à côte (dialogue tronqué, résumé humain, résumé du modèle).

In [ ]:
def mean_words(texts):
    return np.mean([len(t.split()) for t in texts])


# VOTRE CODE ICI

**Question** : quelles différences de **forme** observez-vous entre les résumés du modèle et ceux des humains ? Pourquoi pénalisent-elles le score ROUGE, alors que le résumé du modèle est peut-être correct ? Pourriez-vous les corriger par le prompt seul (pensez au few-shot du TP d'initiation) ? À quel prix ?

✍️ **Votre réponse :**



## 3. Préparer les données

### Le format *prompt / completion*

Pour l'entraînement, chaque exemple est découpé en deux parties :
- **prompt** : ce que le modèle reçoit, exactement comme à l'inférence (chat template compris) ;
- **completion** : ce que le modèle doit apprendre à produire, ici le résumé humain.

C'est le format attendu par le `SFTTrainer` de TRL. Point essentiel : le prompt d'entraînement doit être **identique** à celui utilisé à l'inférence (même instruction, même chat template, même gestion du mode raisonnement), sinon le modèle n'appliquera pas ce qu'il a appris.

**Exercice 4** : écrire `format_example(example)` qui renvoie un dictionnaire `{"prompt": ..., "completion": ...}`. Le prompt est le texte produit par le chat template à partir de `build_messages`, avec `add_generation_prompt=True` et `enable_thinking=False` (comme dans `summarize`). La completion est le résumé humain.

In [ ]:
def format_example(example):
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


formatted = format_example(dataset["train"][0])
print(formatted["prompt"] + formatted["completion"])

**Question** : le prompt se termine par `<think>\n\n</think>\n\n`. D'où vient ce bloc et pourquoi est-il important de le garder à l'entraînement ? Par quel token la séquence d'entraînement doit-elle se terminer, et que se passerait-il à l'inférence s'il était absent ?

✍️ **Votre réponse :**



### Longueur des exemples

Plus les séquences sont longues, plus l'entraînement consomme de mémoire (les activations sont proportionnelles à la longueur) et de temps. Les quelques exemples très longs déterminent souvent le pic de mémoire.

**Exercice 5** : calculer le nombre de tokens de chaque exemple d'entraînement (prompt + completion) sur les 2 000 premiers exemples, tracer l'histogramme et afficher les quantiles à 50 %, 95 % et 99 %. Choisir une valeur de `MAX_LENGTH` qui conserve environ 98 % des exemples.

In [ ]:
sample = dataset["train"].select(range(2000)).map(format_example)
# VOTRE CODE ICI

In [ ]:
MAX_LENGTH = 512  # à ajuster selon votre analyse

Plutôt que de **tronquer** les exemples trop longs (ce qui couperait la fin, c'est-à-dire... le résumé à apprendre !), nous les **retirons**. Nous prenons 1 000 exemples d'entraînement et 100 de validation : c'est peu, mais suffisant pour apprendre un style, et l'entraînement reste d'une dizaine de minutes sur un T4.

In [ ]:
N_TRAIN, N_VALID = 1000, 100


def prepare(split, n):
    ds = dataset[split].shuffle(seed=42).select(range(3 * n)).map(format_example)
    ds = ds.filter(lambda ex: len(tokenizer(ex["prompt"] + ex["completion"], add_special_tokens=False)["input_ids"]) < MAX_LENGTH)
    return ds.select(range(n)).select_columns(["prompt", "completion"])


train_ds = prepare("train", N_TRAIN)
valid_ds = prepare("validation", N_VALID)
train_ds, valid_ds

### Sur quels tokens le modèle apprend-il ?

Un LLM *decoder-only* est entraîné à **prédire le token suivant** : la perte (*cross-entropy*) est calculée à chaque position de la séquence. Pour le fine-tuning d'instructions, on ne veut **pas** que le modèle apprenne à générer le prompt (les dialogues) : on veut qu'il apprenne à produire la réponse **sachant** le prompt.

On fournit donc au modèle, pour chaque exemple :
- `input_ids` : les tokens du prompt suivis de ceux de la completion ;
- `labels` : une copie de `input_ids` où les positions du prompt sont remplacées par **-100**, la valeur que PyTorch ignore dans le calcul de la perte.

(Le décalage d'une position entre entrée et cible est fait automatiquement à l'intérieur du modèle.)

**Exercice 6** : écrire `build_labels(prompt, completion)` qui renvoie `input_ids` et `labels` pour un exemple, en ajoutant le token de fin `tokenizer.eos_token_id` à la fin de la completion. Vérifier en décodant uniquement les tokens dont le label n'est pas -100.

In [ ]:
def build_labels(prompt, completion):
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


input_ids, labels = build_labels(train_ds[0]["prompt"], train_ds[0]["completion"])
print(f"{len(input_ids)} tokens, dont {sum(l != -100 for l in labels)} comptent dans la perte")
print(repr(tokenizer.decode([l for l in labels if l != -100])))

**Question** : sur cet exemple, quelle proportion des tokens contribue à la perte ? Que se passerait-il si l'on calculait la perte sur toute la séquence (prompt compris) ? Et si l'on oubliait les -100 sur les tokens de *padding* ajoutés pour former des lots de même longueur ?

✍️ **Votre réponse :**



## 4. LoRA : *Low-Rank Adaptation*

LoRA gèle chaque matrice de poids $W$ (de taille $d_{out} \times d_{in}$) du modèle et lui ajoute une mise à jour de **rang faible** :

$$W' = W + \frac{\alpha}{r} B A \qquad A \in \mathbb{R}^{r \times d_{in}},\; B \in \mathbb{R}^{d_{out} \times r}$$

Seules $A$ et $B$ sont entraînées. $B$ est initialisée à zéro : au début de l'entraînement, le modèle est **exactement** le modèle d'origine. Les hyperparamètres principaux :
- **`r`** (rang) : la capacité de l'adaptateur. 8 à 32 suffisent pour apprendre un style ou une tâche ;
- **`lora_alpha`** : un facteur d'échelle, souvent fixé à `r` ou `2 * r` ;
- **`target_modules`** : les matrices adaptées. Les études récentes montrent qu'adapter **toutes** les couches linéaires (`"all-linear"` : attention et MLP) donne de meilleurs résultats qu'adapter seulement `q` et `v` comme dans l'article LoRA d'origine (2021).

À la fin, l'adaptateur ne pèse que quelques dizaines de Mo, et on peut en entraîner un par tâche sur le même modèle de base.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
)

**Exercice 7** : calculer **à la main** le nombre de paramètres LoRA ajoutés. Pour une couche linéaire $d_{in} \to d_{out}$, LoRA ajoute $r \times (d_{in} + d_{out})$ paramètres. Les dimensions de chaque couche linéaire d'un bloc sont données par `model.model.layers[0]`, et le nombre de blocs par `model.config.num_hidden_layers`.

In [ ]:
print(model.model.layers[0])

In [ ]:
# VOTRE CODE ICI
print(f"Paramètres LoRA attendus : {expected:,}")

Ajoutons maintenant l'adaptateur au modèle et vérifions votre calcul.

**Attention** : `get_peft_model` **modifie le modèle en place** (les couches LoRA sont injectées dans `model`). Il n'existe donc plus de « modèle d'origine » séparé en mémoire : pour retrouver le comportement du modèle de base, on désactive temporairement l'adaptateur avec `with peft_model.disable_adapter():`. Comparer « le modèle d'origine » et « le modèle fine-tuné » sans en tenir compte est une erreur classique, qui figurait d'ailleurs dans l'ancienne version de ce TP.

In [ ]:
peft_model = get_peft_model(model, lora_config)
peft_model.print_trainable_parameters()

**Question** : quel pourcentage des paramètres est entraîné ? Combien de Mo pèsera l'adaptateur sauvegardé en 16 bits ? Comment évolue le nombre de paramètres si l'on double `r` ?

✍️ **Votre réponse :**



## 5. Entraînement avec TRL

Le `SFTTrainer` de [TRL](https://huggingface.co/docs/trl/sft_trainer) (*Transformer Reinforcement Learning*) est aujourd'hui l'outil standard pour le fine-tuning supervisé. À partir d'un jeu de données *prompt / completion*, il tokenise, ajoute le token de fin, masque le prompt et le padding (ce que vous avez fait à la main), puis entraîne avec le `Trainer` de transformers.

Les hyperparamètres d'entraînement sont regroupés dans un `SFTConfig` :

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir="qwen3-dialogsum-lora",
    # Taille de lot effective = per_device_train_batch_size x gradient_accumulation_steps = 16
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,            # LoRA tolère des taux d'apprentissage plus élevés que le fine-tuning complet
    lr_scheduler_type="cosine",
    warmup_steps=0.05,
    max_length=MAX_LENGTH,
    gradient_checkpointing=True,   # recalcule les activations au lieu de les stocker : moins de mémoire, ~30 % plus lent
    fp16=not USE_BF16,             # précision mixte : float16 sur T4, bfloat16 sur GPU récent
    bf16=USE_BF16,
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=20,
    save_strategy="no",            # nous sauvegardons l'adaptateur nous-mêmes à la fin
    report_to="none",              # pas de Weights & Biases / TensorBoard
    seed=42,
)

trainer = SFTTrainer(
    model=peft_model,
    args=sft_config,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    processing_class=tokenizer,
)

**Exercice 8** : vérifier que TRL a bien fait ce que vous avez codé à l'exercice 6. Le jeu tokenisé est dans `trainer.train_dataset`, avec les colonnes `input_ids` et `labels`. Comparer le premier exemple avec le résultat de `build_labels`, et décoder les tokens qui comptent dans la perte.

In [ ]:
tokenized = trainer.train_dataset[0]
# VOTRE CODE ICI

**Questions** (à traiter pendant l'entraînement) :
1. Combien de **pas d'optimisation** (*steps*) l'entraînement va-t-il faire ? Vérifiez avec la barre de progression.
2. À quoi sert `gradient_accumulation_steps` ? Pourquoi ne pas simplement mettre `per_device_train_batch_size=16` ?
3. Pourquoi évaluer sur un jeu de **validation** pendant l'entraînement, et pas sur le jeu de test ?

✍️ **Votre réponse :**



In [ ]:
torch.cuda.reset_peak_memory_stats()
start = time.time()
trainer.train()
print(f"Durée : {(time.time() - start) / 60:.1f} min | {gpu_memory()}")

**Exercice 9** : tracer sur un même graphique la perte d'entraînement (`loss`) et la perte de validation (`eval_loss`) en fonction du pas (`step`). Ces valeurs sont dans `trainer.state.log_history`, une liste de dictionnaires.

In [ ]:
# VOTRE CODE ICI

**Question** : décrire la courbe. La perte de validation remonte-t-elle (sur-apprentissage) ? Que vaut $e^{\text{perte}}$ (la *perplexité*) au début et à la fin, et comment l'interpréter ?

✍️ **Votre réponse :**



## 6. Évaluation

Une erreur classique est de juger un fine-tuning en regardant **quelques** générations. C'est une mauvaise idée car (1) ce n'est pas quantifié et (2) ce qui fonctionne sur trois exemples peut échouer sur des milliers (généralisation). On compare donc le modèle de base et le modèle fine-tuné **sur le même jeu de test, avec la même métrique et les mêmes paramètres de génération**.

**Exercice 10** : générer les résumés du modèle fine-tuné (`ft_summaries`) sur les dialogues d'évaluation, calculer ses scores ROUGE et construire un tableau comparatif avec ceux du modèle de base. Comparer aussi les longueurs moyennes.

In [ ]:
# VOTRE CODE ICI

In [ ]:
pd.DataFrame({"humain": human_first_refs, "base": base_summaries, "fine-tuné": ft_summaries}).head(5)

**Questions** :
1. Le gain est-il significatif ? Avec 50 exemples seulement, que faudrait-il faire pour s'en assurer ?
2. ROUGE a-t-il augmenté parce que les résumés sont **meilleurs** ou parce qu'ils **ressemblent davantage** aux références ? Comment pourriez-vous mesurer la qualité du contenu (pensez au TP d'initiation) ?

✍️ **Votre réponse :**



### Oubli catastrophique

Le fine-tuning spécialise le modèle. Vérifions s'il sait encore faire **autre chose** que résumer des dialogues. Grâce à LoRA, il suffit de désactiver l'adaptateur pour retrouver le modèle d'origine.

**Exercice 11** : écrire `chat(model, question)` qui génère une réponse (chat template, `enable_thinking=False`, décodage glouton, 150 nouveaux tokens au plus), puis poser les questions ci-dessous au modèle avec et sans adaptateur.

In [ ]:
@torch.no_grad()
def chat(model, question, max_new_tokens=150):
    # VOTRE CODE ICI
    raise NotImplementedError("À compléter")


questions = [
    "Explique en deux phrases ce qu'est le fine-tuning d'un LLM.",
    "Write a short dialogue between a waiter and a customer.",
]
for question in questions:
    print(f"### {question}")
    with peft_model.disable_adapter():
        print(f"--- SANS adaptateur :\n{chat(peft_model, question)}")
    print(f"--- AVEC adaptateur :\n{chat(peft_model, question)}\n")

**Question** : le modèle fine-tuné a-t-il gardé ses capacités générales ? Quels avantages pratiques LoRA offre-t-il face à ce risque ?

✍️ **Votre réponse :**



## 7. Sauvegarder, recharger, fusionner

`save_pretrained` sur un modèle PEFT ne sauvegarde **que l'adaptateur**.

In [ ]:
ADAPTER_DIR = "qwen3-dialogsum-lora/adapter"
peft_model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
for name in sorted(os.listdir(ADAPTER_DIR)):
    print(f"{name:30} {os.path.getsize(os.path.join(ADAPTER_DIR, name)) / 1e6:8.1f} Mo")

Pour réutiliser l'adaptateur (dans une autre session, sur un autre serveur), on charge le modèle de base puis l'adaptateur par-dessus. Libérons d'abord la mémoire du GPU.

**Exercice 12** : supprimer les objets qui occupent le GPU (`trainer`, `peft_model`, `model`) puis appeler `free_memory()`. Recharger ensuite le modèle de base et l'adaptateur avec `PeftModel.from_pretrained`, et vérifier sur 5 dialogues que les résumés sont identiques à `ft_summaries`.

In [ ]:
from peft import PeftModel

# VOTRE CODE ICI

Pour le **déploiement**, on fusionne souvent l'adaptateur dans les poids ($W' = W + \frac{\alpha}{r} B A$ est calculé une fois pour toutes) : on obtient un modèle standard, sans surcoût de calcul à l'inférence, que l'on peut convertir pour [Ollama](https://ollama.com) ou servir avec [vLLM](https://docs.vllm.ai).

In [ ]:
merged = reloaded.merge_and_unload()
print(type(merged).__name__, "| couches LoRA restantes :", sum("lora" in n for n, _ in merged.named_parameters()))
# merged.save_pretrained("qwen3-dialogsum-merged")  # ~3,4 Go : à ne pas faire sur Colab sans raison

**Question** : quels sont les avantages et inconvénients de fusionner l'adaptateur, par rapport à le garder séparé ?

✍️ **Votre réponse :**



**Partager son modèle (optionnel)** : avec un compte Hugging Face gratuit et un *token* d'accès en écriture (stocké dans les *Secrets* de Colab, jamais en clair dans le notebook), `reloaded.push_to_hub("votre-nom/qwen3-dialogsum-lora")` publie l'adaptateur sur le Hub. Pensez à rédiger la *model card* : données, hyperparamètres, scores, limites.

In [ ]:
del merged, reloaded, base_model
free_memory()

## Mini-projets (au choix)

Choisissez **un** des deux mini-projets. Ils réutilisent les fonctions définies plus haut (`format_example`, `prepare`, `summarize`, `rouge_scores`...). Rédigez une courte analyse de vos résultats.

### Mini-projet A : QLoRA sur un modèle plus gros

**QLoRA** ([Dettmers et al., 2023](https://arxiv.org/abs/2305.14314)) combine LoRA et la quantification : les poids gelés du modèle sont stockés en **4 bits** (format NF4), ce qui divise leur taille par 4 par rapport au 16 bits. Les adaptateurs LoRA restent en haute précision. C'est ce qui permet de fine-tuner des modèles de 7-8 milliards de paramètres sur un GPU gratuit.

Travail demandé :
1. Charger [Qwen3-4B](https://huggingface.co/Qwen/Qwen3-4B) en 4 bits avec un `BitsAndBytesConfig` (`load_in_4bit=True`, `bnb_4bit_quant_type="nf4"`, `bnb_4bit_compute_dtype=DTYPE`, `bnb_4bit_use_double_quant=True`). Mesurer la mémoire GPU occupée et la comparer à ce qu'occuperait le modèle en 16 bits.
2. Préparer le modèle avec `peft.prepare_model_for_kbit_training`, puis l'entraîner avec la même configuration LoRA et les mêmes données que dans le TP (pensez à réduire `per_device_train_batch_size` et à augmenter `gradient_accumulation_steps` d'autant si la mémoire manque).
3. Évaluer le modèle 4B de base et le modèle 4B fine-tuné avec ROUGE, puis construire un tableau comparatif : Qwen3-1.7B base, 1.7B LoRA, 4B base, 4B QLoRA, avec le temps d'entraînement et le pic de mémoire.
4. Analyse : un petit modèle fine-tuné peut-il rivaliser avec un modèle plus gros non fine-tuné ? Quel compromis qualité / coût choisiriez-vous en production ?

### Mini-projet B : combien de données faut-il ?

Le coût d'un fine-tuning est souvent dominé par la **constitution des données** (annotation humaine). Il est donc crucial de savoir combien d'exemples sont vraiment nécessaires.

Travail demandé :
1. Écrire une fonction `finetune(n_train, r=16)` qui charge un modèle frais, l'entraîne sur `n_train` exemples et renvoie ses scores ROUGE. Pour que l'étude tienne dans le temps d'une séance, utilisez [Qwen3-0.6B](https://huggingface.co/Qwen/Qwen3-0.6B) (pensez à libérer la mémoire entre deux entraînements).
2. Entraîner avec 0 (modèle de base), 100, 400 et 1 600 exemples et tracer ROUGE-L en fonction du nombre d'exemples (échelle logarithmique).
3. Avec le meilleur nombre d'exemples, comparer `r=4` et `r=64`.
4. Analyse : à partir de combien d'exemples le gain devient-il marginal ? Le rang a-t-il une influence ? Qu'en concluez-vous pour un projet réel ?

## Pour aller plus loin

- **[Unsloth](https://github.com/unslothai/unsloth)** : bibliothèque gratuite qui accélère LoRA / QLoRA d'un facteur 2 environ et réduit la mémoire, avec des notebooks Colab prêts à l'emploi pour les modèles récents (Qwen, Llama, Gemma...). Elle s'appuie sur TRL : ce que vous avez appris ici s'y transpose directement.
- **Alignement par préférences** : après le SFT, on peut entraîner le modèle à préférer certaines réponses à d'autres avec **DPO** (`DPOTrainer` de TRL, sur des paires réponse choisie / réponse rejetée), ou par apprentissage par renforcement avec **GRPO** (`GRPOTrainer`), la méthode popularisée par DeepSeek-R1 pour entraîner les modèles de raisonnement.
- **Déploiement local** : après fusion, convertir le modèle au format GGUF avec [llama.cpp](https://github.com/ggml-org/llama.cpp) pour l'exécuter avec Ollama ou LM Studio sur un ordinateur portable.